# Optimisation des Achats — Réduction de la Valeur de Stock
## Horizon fiscal Mai → Septembre | 15 Références | MILP Pyomo

---

## 1. Contexte et problématique

Nous sommes en **mai** : il reste **5 mois** jusqu'à la clôture de l'année fiscale (fin septembre). La valeur totale du stock actuel est d'environ **4 000 000 €**, bien au-dessus de la **cible fiscale de 2 200 000 €**.

L'objectif est de déterminer, pour chacune des 15 références, **quand et combien commander** sur les 5 mois restants, de façon à :

- ✅ Satisfaire la demande client à chaque période
- ✅ Respecter un **stock de sécurité minimum** (taux de service 95 %) à partir de juin
- ✅ Respecter la **quantité minimale de commande** ($MOQ_i$) par fournisseur
- ✅ Tenir compte du **délai de livraison** ($LT_i$)
- ✅ **Aucune commande en août** (fermeture fournisseurs)
- ✅ Respecter la **capacité de stockage** physique ($V_{max}$ m³)
- 📉 **Ramener la valeur du stock à 2 200 000 € ou moins en fin septembre**
- 📉 **Minimiser le coût total** = coût de possession + coût fixe de commande

---

## 2. Formulation mathématique

### Ensembles et indices

| Symbole | Signification |
|---|---|
| $I = \{1,\ldots,15\}$ | Ensemble des références produit |
| $T = \{1,\ldots,5\}$ | Périodes : 1=Mai, 2=Juin, 3=Juillet, 4=Août, 5=Septembre |
| $T_{août} \subset T$ | Sous-ensemble des périodes de fermeture fournisseurs (ici $\{4\}$) |

### Paramètres

| Symbole | Signification |
|---|---|
| $p_i$ | Prix unitaire du PN $i$ (€/unité) |
| $D_{i,t}$ | Demande du PN $i$ en période $t$ (unités) |
| $S_{i,0}$ | Stock initial du PN $i$ (t = 0, fin avril) |
| $LT_i$ | Délai de livraison fournisseur (mois) |
| $MOQ_i$ | Quantité minimale de commande (unités) |
| $SS_i = z_\alpha \cdot \sigma_{D,i} \cdot \sqrt{LT_i}$ | Stock de sécurité (formule EOQ) |
| $c^s_i = \frac{\tau}{12} \cdot p_i$ | Coût de stockage unitaire mensuel ($\tau = 20\,\%$/an) |
| $c^o_i$ | Coût fixe de passation de commande (€/commande) |
| $v_i$ | Volume unitaire de stockage (m³/unité) |
| $V_{max}$ | Capacité totale de stockage (m³) |
| $TARGET$ | Cible de valeur de stock en fin d'horizon (€) |
| $M$ | Grand nombre (*big-M* pour linéarisation) |

### Variables de décision

| Variable | Domaine | Signification |
|---|---|---|
| $S_{i,t}$ | $\mathbb{R}^+$ | Stock du PN $i$ en fin de période $t$ |
| $O_{i,t}$ | $\mathbb{R}^+$ | Quantité commandée du PN $i$ en période $t$ |
| $z_{i,t}$ | $\{0,1\}$ | 1 si une commande est passée pour le PN $i$ en $t$ |

### Fonction objectif

$$\min\; Z = \underbrace{\sum_{i \in I}\sum_{t \in T} c^s_i \cdot S_{i,t}}_{\text{coût de possession du stock}} + \underbrace{\sum_{i \in I}\sum_{t \in T} c^o_i \cdot z_{i,t}}_{\text{coût de passation des commandes}}$$

### Contraintes

$$\text{(C1 — Bilan)}\quad S_{i,t} = S_{i,t-1} + O_{i,\;t-LT_i} - D_{i,t} \quad \forall i \in I,\; t \in T$$

> Convention : $S_{i,0} = $ stock initial ; si $t - LT_i < 1$ alors pas de réception ($O_{i,t-LT_i} = 0$).

$$\text{(C2a — MOQ min)}\quad O_{i,t} \geq MOQ_i \cdot z_{i,t} \quad \forall i, t$$

$$\text{(C2b — big-M)}\quad O_{i,t} \leq M \cdot z_{i,t} \quad \forall i, t$$

$$\text{(C3 — Stock de sécurité)}\quad S_{i,t} \geq SS_i \quad \forall i,\; t \geq 2$$

$$\text{(C4 — Fermeture août)}\quad O_{i,t} = 0 \quad \forall i \in I,\; t \in T_{août}$$

$$\text{(C5 — Capacité de stockage)}\quad \sum_{i \in I} v_i \cdot S_{i,t} \leq V_{max} \quad \forall t \in T$$

$$\text{(C6 — Cible fiscale fin septembre)}\quad \sum_{i \in I} p_i \cdot S_{i,T} \leq TARGET$$


---
## 3. Implémentation — même démarche que la méthode AMPL/GUSEK

### Étape 0 — Imports

In [ ]:
from pyomo.environ import *
from IPython.display import display
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import shutil, subprocess

# Installation CBC si nécessaire (Google Colab)
if shutil.which("cbc") is None:
    subprocess.run(["apt-get", "install", "-y", "coinor-cbc"], check=True,
                   capture_output=True)
print(f"✅  CBC : {shutil.which('cbc')}")


---
### Étape 1 — Les Paramètres (modèle abstrait Pyomo)

In [ ]:
model = AbstractModel()

# ── Ensembles ────────────────────────────────────────────────
model.I       = Set()
model.T       = Param(within=PositiveIntegers)
model.PER     = RangeSet(1, model.T)
model.AoutSet = Set(within=model.PER)           # périodes de fermeture fournisseurs

# ── Paramètres scalaires ─────────────────────────────────────
model.TARGET  = Param()     # cible valeur de stock fin d'horizon (€)
model.V_max   = Param()     # capacité de stockage globale (m³)
model.M       = Param()     # big-M

# ── Paramètres indexés par référence ─────────────────────────
model.prix          = Param(model.I)   # p_i  : prix unitaire (€/u)
model.stock_init    = Param(model.I)   # S0_i : stock initial (u)
model.LT            = Param(model.I)   # LT_i : délai de livraison (mois)
model.MOQ           = Param(model.I)   # MOQ_i : quantité minimale de commande
model.volume        = Param(model.I)   # v_i  : volume unitaire (m³/u)
model.SS            = Param(model.I)   # SS_i : stock de sécurité (u)
model.cout_stockage = Param(model.I)   # c^s_i : coût stockage mensuel (€/u/mois)
model.cout_commande = Param(model.I)   # c^o_i : coût passation commande (€)

# ── Paramètres indexés par référence × période ───────────────
model.demande = Param(model.I, model.PER)   # D_{i,t} : demande (u)

print("✅  Étape 1 — Paramètres abstraits déclarés")


---
### Étape 2 — Les Variables

In [ ]:
model.S = Var(model.I, model.PER, domain=NonNegativeReals)   # S_{i,t} : stock fin période
model.O = Var(model.I, model.PER, domain=NonNegativeReals)   # O_{i,t} : quantité commandée
model.z = Var(model.I, model.PER, domain=Binary)             # z_{i,t} : décision commande

print("✅  Étape 2 — Variables déclarées")
print("   S[i,t] : continue ≥ 0  (stock fin de période)")
print("   O[i,t] : continue ≥ 0  (quantité commandée)")
print("   z[i,t] : binaire {0,1}  (commande passée ?)")


---
### Étape 3 — La Fonction Objectif et les Contraintes C1 → C6

In [ ]:
# ── Fonction objectif ──────────────────────────────────────────────────────
def cost_rule(m):
    return (sum(m.cout_stockage[i] * m.S[i,t] for i in m.I for t in m.PER)
          + sum(m.cout_commande[i] * m.z[i,t]  for i in m.I for t in m.PER))
model.cost = Objective(rule=cost_rule, sense=minimize)

# ── C1 — Bilan de stock ────────────────────────────────────────────────────
def c1_rule(m, i, t):
    S_prev    = m.stock_init[i] if t == 1 else m.S[i, t-1]
    t_cmd     = t - m.LT[i]
    reception = m.O[i, t_cmd] if t_cmd >= 1 else 0
    return m.S[i,t] == S_prev + reception - m.demande[i,t]
model.C1_Bilan = Constraint(model.I, model.PER, rule=c1_rule)

# ── C2a — MOQ minimum ──────────────────────────────────────────────────────
def c2a_rule(m, i, t):
    return m.O[i,t] >= m.MOQ[i] * m.z[i,t]
model.C2a_MOQ_min = Constraint(model.I, model.PER, rule=c2a_rule)

# ── C2b — big-M (si z=0 alors O=0) ────────────────────────────────────────
def c2b_rule(m, i, t):
    return m.O[i,t] <= m.M * m.z[i,t]
model.C2b_MOQ_max = Constraint(model.I, model.PER, rule=c2b_rule)

# ── C3 — Stock de sécurité (à partir de t=2) ───────────────────────────────
def c3_rule(m, i, t):
    if t == 1:
        return Constraint.Skip
    return m.S[i,t] >= m.SS[i]
model.C3_StockSecurite = Constraint(model.I, model.PER, rule=c3_rule)

# ── C4 — Fermeture fournisseurs en août ────────────────────────────────────
def c4_rule(m, i, t):
    return m.O[i,t] == 0
model.C4_FermetureFournisseur = Constraint(model.I, model.AoutSet, rule=c4_rule)

# ── C5 — Capacité de stockage ──────────────────────────────────────────────
def c5_rule(m, t):
    return sum(m.volume[i] * m.S[i,t] for i in m.I) <= m.V_max
model.C5_CapaciteStockage = Constraint(model.PER, rule=c5_rule)

# ── C6 — Cible de valeur de stock en fin d'horizon ────────────────────────
def c6_rule(m):
    T_fin = value(m.T)
    return sum(m.prix[i] * m.S[i, T_fin] for i in m.I) <= m.TARGET
model.C6_Cible = Constraint(rule=c6_rule)

print("✅  Étape 3 — Objectif + Contraintes C1→C6 définis")


---
### Étape 4 — Résolution

In [ ]:
def resoudre(instance, solveur="cbc", verbose=False):
    opt = SolverFactory(solveur)
    return opt.solve(instance, tee=verbose)

print("✅  Étape 4 — Fonction resoudre() définie")


---
### Étape 5 — Mise en place des données

#### 15 références réparties en 3 groupes

| Groupe | Références | Rotation | Comportement attendu |
|--------|-----------|----------|---------------------|
| **A — Haute rotation** | PN-01 à PN-07 | 35–45 %/mois | Commandes multiples obligatoires (stock épuisé sans réapprovisionnement) |
| **B — Rotation moyenne** | PN-08 à PN-11 | 18–25 %/mois | 1 à 2 commandes selon la période |
| **C — Faible rotation** | PN-12 à PN-15 | 5–8 %/mois | Aucune commande (contribution à la réduction du stock) |

Stock initial total calibré à **≈ 4 000 000 €** — cible fin septembre : **2 200 000 €**.


In [ ]:
# ══════════════════════════════════════════════════════════════
# PARAMÈTRES DE L'HORIZON
# ══════════════════════════════════════════════════════════════
noms_mois = {1:"Mai", 2:"Juin", 3:"Juil.", 4:"Août", 5:"Sept."}
T_max     = 5
t_aout    = [4]     # fermeture fournisseurs en août

# ══════════════════════════════════════════════════════════════
# RÉFÉRENCES — prix, délai, MOQ, coût commande, volume
# ══════════════════════════════════════════════════════════════
references = {
    # ── Groupe A : Haute rotation (commandes obligatoires) ──────
    "PN-01": {"prix":200, "LT":1, "MOQ":500,  "cout_commande":300, "volume":0.06},
    "PN-02": {"prix":100, "LT":1, "MOQ":1000, "cout_commande":180, "volume":0.03},
    "PN-03": {"prix":400, "LT":1, "MOQ":200,  "cout_commande":500, "volume":0.09},
    "PN-04": {"prix":150, "LT":1, "MOQ":500,  "cout_commande":250, "volume":0.05},
    "PN-05": {"prix":600, "LT":1, "MOQ":100,  "cout_commande":700, "volume":0.12},
    "PN-06": {"prix": 80, "LT":2, "MOQ":2000, "cout_commande":150, "volume":0.03},
    "PN-07": {"prix":350, "LT":2, "MOQ":200,  "cout_commande":450, "volume":0.08},
    # ── Groupe B : Rotation moyenne (quelques commandes) ────────
    "PN-08": {"prix":120, "LT":1, "MOQ":600,  "cout_commande":200, "volume":0.04},
    "PN-09": {"prix":250, "LT":1, "MOQ":300,  "cout_commande":380, "volume":0.07},
    "PN-10": {"prix": 60, "LT":1, "MOQ":2000, "cout_commande":120, "volume":0.02},
    "PN-11": {"prix":180, "LT":2, "MOQ":400,  "cout_commande":280, "volume":0.06},
    # ── Groupe C : Faible rotation (aucune commande prévue) ─────
    "PN-12": {"prix": 30, "LT":1, "MOQ":3000, "cout_commande": 80, "volume":0.01},
    "PN-13": {"prix": 12, "LT":1, "MOQ":8000, "cout_commande": 60, "volume":0.005},
    "PN-14": {"prix": 50, "LT":1, "MOQ":2000, "cout_commande": 90, "volume":0.015},
    "PN-15": {"prix": 90, "LT":1, "MOQ":1000, "cout_commande":150, "volume":0.025},
}
refs = list(references.keys())

# ══════════════════════════════════════════════════════════════
# STOCK INITIAL (calibré pour valeur totale ≈ 4 000 000 €)
# ── Groupe A ─────────────────────────────────────────────────
# PN-01 : 3 500 × 200 = 700 000 €
# PN-02 : 3 000 × 100 = 300 000 €
# PN-03 :   600 × 400 = 240 000 €
# PN-04 : 1 800 × 150 = 270 000 €
# PN-05 :   400 × 600 = 240 000 €
# PN-06 : 3 500 ×  80 = 280 000 €
# PN-07 :   700 × 350 = 245 000 €  ─── sous-total A = 2 275 000 €
# ── Groupe B ─────────────────────────────────────────────────
# PN-08 : 2 000 × 120 = 240 000 €
# PN-09 :   800 × 250 = 200 000 €
# PN-10 : 3 500 ×  60 = 210 000 €
# PN-11 : 1 300 × 180 = 234 000 €  ─── sous-total B =   884 000 €
# ── Groupe C ─────────────────────────────────────────────────
# PN-12 : 9 000 ×  30 = 270 000 €
# PN-13 :22 000 ×  12 = 264 000 €
# PN-14 : 5 000 ×  50 = 250 000 €
# PN-15 : 3 000 ×  90 = 270 000 €  ─── sous-total C = 1 054 000 €
# ─────────────────────────── TOTAL   ≈   4 213 000 €
# ══════════════════════════════════════════════════════════════
stock_init = {
    "PN-01":3500, "PN-02":3000, "PN-03":600,  "PN-04":1800, "PN-05":400,
    "PN-06":3500, "PN-07":700,
    "PN-08":2000, "PN-09":800,  "PN-10":3500, "PN-11":1300,
    "PN-12":9000, "PN-13":22000,"PN-14":5000, "PN-15":3000,
}

valeur_initiale = sum(stock_init[i]*references[i]["prix"] for i in refs)
TARGET = 2_200_000

print("=" * 58)
print("  DONNÉES FINANCIÈRES DE DÉPART")
print("=" * 58)
for grp, pns in [("A (Haute rot.)",refs[:7]),
                 ("B (Moy. rot.)", refs[7:11]),
                 ("C (Faible rot.)",refs[11:])]:
    v = sum(stock_init[i]*references[i]["prix"] for i in pns)
    print(f"  Groupe {grp:<18} : {v:>12,.0f} €")
print(f"  {'─'*42}")
print(f"  Valeur initiale totale  : {valeur_initiale:>12,.0f} €")
print(f"  Cible fin septembre     : {TARGET:>12,.0f} €")
print(f"  Réduction à atteindre   : {valeur_initiale-TARGET:>12,.0f} €  "
      f"({(valeur_initiale-TARGET)/valeur_initiale*100:.1f} %)")
print("=" * 58)


In [ ]:
# ══════════════════════════════════════════════════════════════
# DEMANDE MENSUELLE D[i,t]
# Taux de consommation calibré par groupe + saisonnalité légère
# ══════════════════════════════════════════════════════════════
facteur = {
    # Groupe A : 35–45 % du stock initial / mois
    "PN-01":0.40, "PN-02":0.38, "PN-03":0.42, "PN-04":0.39,
    "PN-05":0.44, "PN-06":0.36, "PN-07":0.41,
    # Groupe B : 18–25 %
    "PN-08":0.22, "PN-09":0.20, "PN-10":0.24, "PN-11":0.19,
    # Groupe C : 5–8 %
    "PN-12":0.07, "PN-13":0.06, "PN-14":0.08, "PN-15":0.07,
}
variation = {1:0.95, 2:1.00, 3:1.05, 4:1.00, 5:1.00}

demande = {}
for i in refs:
    base = round(facteur[i] * stock_init[i])
    for t in range(1, T_max+1):
        demande[(i,t)] = round(base * variation[t])

df_dem = pd.DataFrame(
    {noms_mois[t]: {i: demande[(i,t)] for i in refs} for t in range(1,T_max+1)}
)
df_dem.index.name = "PN"
print("Demande mensuelle D[i,t] (unités) :")
display(df_dem)


In [ ]:
\
# ══════════════════════════════════════════════════════════════
# STOCK DE SÉCURITÉ  SS_i = z_alpha × sigma_{D,i} × sqrt(LT_i)
# z_alpha = 1.65 → niveau de service α = 95 %
# ══════════════════════════════════════════════════════════════
z_alpha = 1.65
sigma_D = {i: np.std([demande[(i,t)] for t in range(1,T_max+1)]) for i in refs}
SS = {i: max(round(z_alpha * sigma_D[i] * math.sqrt(references[i]["LT"])), 1)
      for i in refs}

df_ss = pd.DataFrame({
    "Stock init. (u)": stock_init,
    "Demande moy. (u/mois)": {i: round(np.mean([demande[(i,t)] for t in range(1,T_max+1)])) for i in refs},
    "σ_D": {i: round(sigma_D[i],1) for i in refs},
    "LT (mois)": {i: references[i]["LT"] for i in refs},
    "SS (u)": SS,
    "Prix (€/u)": {i: references[i]["prix"] for i in refs},
}).rename_axis("PN")
print("Stock de sécurité par référence :")
display(df_ss)


In [ ]:
# ══════════════════════════════════════════════════════════════
# PARAMÈTRES CALCULÉS : coût stockage, capacité, big-M
# ══════════════════════════════════════════════════════════════
tau           = 0.20   # taux de possession annuel
cout_stockage = {i: round(tau * references[i]["prix"] / 12, 4) for i in refs}

volume_initial = sum(stock_init[i]*references[i]["volume"] for i in refs)
V_max          = round(volume_initial * 1.25)   # marge de 25 % sur le volume initial

M_bigM = sum(demande[(i,t)] for i in refs for t in range(1,T_max+1)) * 2

print(f"Taux de possession τ      : {tau*100:.0f} %/an")
print(f"Volume stock initial       : {volume_initial:,.1f} m³")
print(f"Capacité stockage V_max    : {V_max:,.0f} m³  (×1.25)")
print(f"Big-M                      : {M_bigM:,.0f}")


In [ ]:
# ══════════════════════════════════════════════════════════════
# CONSTRUCTION DU DataPortal ET DE L'INSTANCE PYOMO
# ══════════════════════════════════════════════════════════════
data = DataPortal()
data['T']       = {None: T_max}
data['AoutSet'] = t_aout
data['I']       = refs
data['M']       = {None: M_bigM}
data['TARGET']  = {None: TARGET}
data['V_max']   = {None: V_max}

data['prix']          = {i: references[i]["prix"] for i in refs}
data['stock_init']    = stock_init
data['LT']            = {i: references[i]["LT"]  for i in refs}
data['MOQ']           = {i: references[i]["MOQ"]  for i in refs}
data['volume']        = {i: references[i]["volume"] for i in refs}
data['SS']            = SS
data['cout_stockage'] = cout_stockage
data['cout_commande'] = {i: references[i]["cout_commande"] for i in refs}
data['demande']       = demande

instance = model.create_instance(data)

n_var  = sum(1 for _ in instance.component_data_objects(Var,        active=True))
n_cst  = sum(1 for _ in instance.component_data_objects(Constraint, active=True))
n_bin  = sum(1 for v in instance.component_data_objects(Var, active=True)
             if v.domain is Binary)
print(f"✅  Instance créée — {len(refs)} références, {T_max} périodes")
print(f"   Variables continues  : {n_var - n_bin}")
print(f"   Variables binaires   : {n_bin}")
print(f"   Contraintes totales  : {n_cst}")


---
## 4. Résolution et résultats

### 4.1 — Lancement du solveur CBC

In [ ]:
resultats = resoudre(instance, solveur="cbc", verbose=False)
statut    = str(resultats.solver.termination_condition)
Z_star    = value(instance.cost)

print("=" * 55)
print("  RÉSULTAT DE LA RÉSOLUTION")
print("=" * 55)
print(f"  Statut solveur   : {statut}")
print(f"  Coût total Z*    : {Z_star:>14,.2f}  €")
print("=" * 55)


### 4.2 — Vérification de la cible fiscale (C6)

In [ ]:
valeur_finale = sum(references[i]["prix"] * value(instance.S[i, T_max])
                    for i in refs)
print(f"  Valeur stock fin mai (t=0) : {valeur_initiale:>14,.0f}  €")
print(f"  Valeur stock fin sept (t=5): {valeur_finale:>14,.0f}  €")
print(f"  Cible fiscale TARGET       : {TARGET:>14,.0f}  €")
print(f"  Réduction obtenue          : {valeur_initiale-valeur_finale:>14,.0f}  €  "
      f"({(valeur_initiale-valeur_finale)/valeur_initiale*100:.1f} %)")
print()
if valeur_finale <= TARGET:
    print("  ✅  Contrainte C6 respectée — Cible fiscale atteinte !")
else:
    print(f"  ❌  Dépassement : {valeur_finale-TARGET:,.0f} €")


### 4.3 — Plan de commandes optimal

In [ ]:
# ── Tableau synthétique : quand et combien commander ──────────────────────
lignes_plan = []
for i in refs:
    row = {"PN": i, "Groupe": "A" if i<=refs[6] else ("B" if i<=refs[10] else "C"),
           "Prix (€)": references[i]["prix"],
           "LT": references[i]["LT"],
           "MOQ": references[i]["MOQ"]}
    total_cmd = 0
    nb_cmd    = 0
    for t in range(1, T_max+1):
        zi = int(round(value(instance.z[i,t])))
        oi = round(value(instance.O[i,t]))
        row[noms_mois[t]] = f"🛒 {oi:,}" if zi else "—"
        total_cmd += oi
        nb_cmd    += zi
    row["Nb cmd"] = nb_cmd
    row["Total commandé (u)"] = total_cmd
    lignes_plan.append(row)

df_plan = pd.DataFrame(lignes_plan).set_index("PN")
print("Plan de commandes optimal (🛒 = commande passée) :")
display(df_plan)

# Résumé
print(f"\nNombre de commandes passées : "
      f"{sum(int(round(value(instance.z[i,t]))) for i in refs for t in range(1,T_max+1))}")
pns_cmd = [i for i in refs if any(int(round(value(instance.z[i,t])))>0 for t in range(1,T_max+1))]
pns_ncmd= [i for i in refs if i not in pns_cmd]
print(f"Références avec commandes   : {pns_cmd}")
print(f"Références sans commande    : {pns_ncmd}")


### 4.4 — Évolution des stocks par référence

In [ ]:
lignes_stock = []
for i in refs:
    row = {"PN": i, "Stock init.": stock_init[i], "SS": SS[i]}
    for t in range(1, T_max+1):
        row[noms_mois[t]] = round(value(instance.S[i,t]))
    row["Val. finale (€)"] = round(references[i]["prix"]*value(instance.S[i,T_max]))
    lignes_stock.append(row)

df_stock = pd.DataFrame(lignes_stock).set_index("PN")
print("Évolution des stocks S[i,t] (unités, fin de période) :")
display(df_stock)


---
## 5. Visualisations

### 5.1 — Trajectoire de la valeur totale du stock

In [ ]:
val_par_mois = []
for t in range(1, T_max+1):
    v = sum(references[i]["prix"]*value(instance.S[i,t]) for i in refs)
    val_par_mois.append(v)

# Décomposition par groupe
val_A = [sum(references[i]["prix"]*value(instance.S[i,t]) for i in refs[:7])  for t in range(1,T_max+1)]
val_B = [sum(references[i]["prix"]*value(instance.S[i,t]) for i in refs[7:11]) for t in range(1,T_max+1)]
val_C = [sum(references[i]["prix"]*value(instance.S[i,t]) for i in refs[11:]) for t in range(1,T_max+1)]

mois_ax    = [noms_mois[t] for t in range(1, T_max+1)]
mois_ax0   = ["Avril
(t=0)"] + mois_ax
val_total0 = [valeur_initiale] + val_par_mois

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Graphe gauche : courbe totale ─────────────────────────────
ax = axes[0]
ax.plot(mois_ax0, val_total0, marker='o', lw=2.5, color='steelblue',
        ms=8, label='Valeur totale du stock')
ax.fill_between(mois_ax0, val_total0, alpha=0.12, color='steelblue')
ax.axhline(valeur_initiale, color='grey',  ls=':', lw=1.5,
           label=f"Stock initial ({valeur_initiale/1e6:.2f} M€)")
ax.axhline(TARGET, color='red', ls='--', lw=2.0,
           label=f"Cible TARGET ({TARGET/1e6:.2f} M€)")
# Annotation valeur finale
ax.annotate(f"Fin sept.
{val_par_mois[-1]/1e6:.2f} M€ ✅",
            xy=("Sept.", val_par_mois[-1]),
            xytext=(-60, 30), textcoords='offset points',
            fontsize=9, color='steelblue', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='steelblue'))
ax.set_title("Trajectoire de la valeur totale du stock", fontsize=11, fontweight='bold')
ax.set_xlabel("Période"); ax.set_ylabel("Valeur du stock (€)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x/1e6:.2f} M€"))
ax.legend(fontsize=9); ax.grid(alpha=0.25)

# ── Graphe droit : aires empilées par groupe ───────────────────
ax2 = axes[1]
ax2.stackplot(mois_ax,
              val_A, val_B, val_C,
              labels=['Groupe A — Haute rotation',
                      'Groupe B — Rotation moyenne',
                      'Groupe C — Faible rotation'],
              colors=['#e74c3c','#f39c12','#27ae60'],
              alpha=0.75)
ax2.axhline(TARGET, color='black', ls='--', lw=2.0,
            label=f"Cible ({TARGET/1e6:.2f} M€)")
ax2.set_title("Décomposition de la valeur par groupe", fontsize=11, fontweight='bold')
ax2.set_xlabel("Période"); ax2.set_ylabel("Valeur du stock (€)")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x/1e6:.2f} M€"))
ax2.legend(fontsize=9, loc='upper right'); ax2.grid(alpha=0.25)

plt.suptitle("Optimisation des Achats — Réduction de Stock Mai → Septembre",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 5.2 — Plan de commandes visuel (carte de chaleur)

In [ ]:
# Heatmap des commandes z[i,t]
Z_mat = np.array([[int(round(value(instance.z[i,t])))
                   for t in range(1,T_max+1)] for i in refs])
O_mat = np.array([[round(value(instance.O[i,t]))
                   for t in range(1,T_max+1)] for i in refs])

fig, ax = plt.subplots(figsize=(9, 7))
cmap = plt.cm.RdYlGn
im = ax.imshow(Z_mat, cmap=cmap, aspect='auto', vmin=0, vmax=1)

# Annotations : quantité commandée si z=1
for ii, i in enumerate(refs):
    for jj, t in enumerate(range(1, T_max+1)):
        zi = Z_mat[ii, jj]
        oi = O_mat[ii, jj]
        if zi:
            ax.text(jj, ii, f"{oi:,}", ha='center', va='center',
                    fontsize=8, fontweight='bold', color='white')
        else:
            ax.text(jj, ii, "—", ha='center', va='center',
                    fontsize=9, color='grey')

ax.set_xticks(range(T_max))
ax.set_xticklabels([noms_mois[t] for t in range(1,T_max+1)], fontsize=10)
ax.set_yticks(range(len(refs)))
ax.set_yticklabels([f"{i}  (LT={references[i]['LT']}m | {references[i]['prix']}€/u)"
                    for i in refs], fontsize=9)

# Séparateurs de groupes
ax.axhline(6.5, color='black', lw=1.5, ls='--')
ax.axhline(10.5, color='black', lw=1.5, ls='--')
ax.axvline(2.5, color='orange', lw=2, ls='--', alpha=0.6)  # avant août

# Annotations groupes
for y, txt in [(3, "Groupe A
Haute rotation"), (8.5, "Groupe B
Moy. rot."), (12.5, "Groupe C
Faible rot.")]:
    ax.text(T_max+0.15, y, txt, va='center', fontsize=8,
            color='black', style='italic')

ax.set_title("Plan de commandes optimal — Quantités commandées O[i,t]
"
             "(Vert = commande passée | Fond rouge = pas de commande | "
             "Ligne orange = avant fermeture août)",
             fontsize=10, fontweight='bold')
ax.set_xlabel("Période")
plt.tight_layout()
plt.show()


### 5.3 — Évolution du stock par référence (top 10)

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 14))
axes = axes.flatten()
colors_grp = {'A':'#e74c3c', 'B':'#f39c12', 'C':'#27ae60'}

for idx, i in enumerate(refs[:12]):
    ax = axes[idx]
    grp = 'A' if idx < 7 else ('B' if idx < 11 else 'C')
    col = colors_grp[grp]

    s_vals = [stock_init[i]] + [round(value(instance.S[i,t])) for t in range(1,T_max+1)]
    ax.plot(mois_ax0, s_vals, marker='o', color=col, lw=2)
    ax.fill_between(mois_ax0, s_vals, alpha=0.15, color=col)
    ax.axhline(SS[i], color='red', ls='--', lw=1.2, label=f"SS={SS[i]:,}u")

    # Marqueurs de commandes
    for t in range(1, T_max+1):
        if int(round(value(instance.z[i,t]))) == 1:
            oi = int(round(value(instance.O[i,t])))
            ax.annotate(f"🛒{oi:,}",
                        xy=(noms_mois[t], s_vals[t]),
                        xytext=(0, 12), textcoords='offset points',
                        ha='center', fontsize=7.5, color='darkgreen', fontweight='bold',
                        arrowprops=dict(arrowstyle='->', color='green', lw=1.0))

    ax.set_title(f"{i} | Grp {grp} | {references[i]['prix']}€/u | LT={references[i]['LT']}m",
                 fontsize=9, fontweight='bold', color=col)
    ax.set_ylabel("Unités", fontsize=8)
    ax.tick_params(axis='x', labelsize=7); ax.tick_params(axis='y', labelsize=8)
    ax.legend(fontsize=8); ax.grid(alpha=0.25)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))

for idx in range(12, len(axes)): axes[idx].set_visible(False)
fig.suptitle("Évolution des stocks par référence (PN-01 à PN-12)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 6. Analyse et interprétation

### 6.1 — Décomposition du coût total

In [ ]:
# ── Coûts par référence et par type ───────────────────────────
lignes_cout = []
for i in refs:
    grp = "A" if i <= refs[6] else ("B" if i <= refs[10] else "C")
    cs  = sum(cout_stockage[i]*value(instance.S[i,t]) for t in range(1,T_max+1))
    co  = sum(references[i]["cout_commande"]*value(instance.z[i,t]) for t in range(1,T_max+1))
    nbc = int(round(sum(value(instance.z[i,t]) for t in range(1,T_max+1))))
    lignes_cout.append({
        "PN": i, "Groupe": grp,
        "Coût possession (€)": round(cs),
        "Coût commande (€)":   round(co),
        "Nb commandes":        nbc,
        "Coût total (€)":      round(cs+co),
        "Val. stock finale (€)": round(references[i]["prix"]*value(instance.S[i,T_max])),
    })

df_cout = pd.DataFrame(lignes_cout).set_index("PN")
df_cout.loc["TOTAL"] = df_cout.sum(numeric_only=True)
df_cout.loc["TOTAL", "Groupe"] = "—"
print("Décomposition du coût total par référence :")
display(df_cout.applymap(lambda x: f"{x:,.0f}" if isinstance(x,(int,float)) else x))


### 6.2 — Analyse de la contribution à la réduction du stock

In [ ]:
lignes_red = []
for i in refs:
    grp   = "A" if i <= refs[6] else ("B" if i <= refs[10] else "C")
    v_ini = references[i]["prix"] * stock_init[i]
    v_fin = references[i]["prix"] * value(instance.S[i,T_max])
    red   = v_ini - v_fin
    lignes_red.append({
        "PN": i, "Groupe": grp,
        "Val. initiale (€)": round(v_ini),
        "Val. finale (€)":   round(v_fin),
        "Réduction (€)":     round(red),
        "Réduction (%)":     round(red/v_ini*100, 1) if v_ini > 0 else 0,
    })

df_red = pd.DataFrame(lignes_red).sort_values("Réduction (€)", ascending=False)
df_red_disp = df_red.copy().set_index("PN")
df_red_disp.loc["TOTAL"] = {
    "Groupe": "—",
    "Val. initiale (€)": round(valeur_initiale),
    "Val. finale (€)":   round(valeur_finale),
    "Réduction (€)":     round(valeur_initiale - valeur_finale),
    "Réduction (%)":     round((valeur_initiale-valeur_finale)/valeur_initiale*100, 1)
}
print("Contribution à la réduction de stock par référence :")
display(df_red_disp.applymap(lambda x: f"{x:,.0f}" if isinstance(x,(int,float)) else x))

# Graphique waterfall simplifié
fig, ax = plt.subplots(figsize=(12, 5))
pns_sorted = df_red["PN"].tolist()
reds       = df_red["Réduction (€)"].tolist()
grps       = df_red["Groupe"].tolist()
col_map    = {"A":"#e74c3c", "B":"#f39c12", "C":"#27ae60"}
bar_colors = [col_map[g] for g in grps]
bars = ax.bar(pns_sorted, reds, color=bar_colors, edgecolor='white')
ax.bar_label(bars, fmt=lambda x: f"{x/1e3:.0f}k", fontsize=8, padding=3)
ax.set_title("Réduction de valeur de stock par référence (€)", fontweight='bold')
ax.set_xlabel("Référence"); ax.set_ylabel("Réduction (€)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x/1e3:.0f}k€"))
legend_patches = [mpatches.Patch(color=col_map[g], label=f"Groupe {g}")
                  for g in ["A","B","C"]]
ax.legend(handles=legend_patches)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()


### 6.3 — Résumé exécutif (vue synthétique pour la soutenance)

In [ ]:
cout_poss = sum(cout_stockage[i]*value(instance.S[i,t])
                for i in refs for t in range(1,T_max+1))
cout_cmds = sum(references[i]["cout_commande"]*value(instance.z[i,t])
                for i in refs for t in range(1,T_max+1))
n_cmd_tot = int(round(sum(value(instance.z[i,t]) for i in refs for t in range(1,T_max+1))))

print()
print("╔" + "═"*56 + "╗")
print("║        RÉSUMÉ EXÉCUTIF — PLAN D'OPTIMISATION          ║")
print("╠" + "═"*56 + "╣")
print(f"║  Références analysées          : {len(refs):>20}   ║")
print(f"║  Horizon d'optimisation        : {'Mai → Septembre (5 mois)':>20}   ║")
print("╠" + "═"*56 + "╣")
print(f"║  Valeur stock initiale (mai)   : {valeur_initiale:>16,.0f} €  ║")
print(f"║  Valeur stock finale (sept.)   : {valeur_finale:>16,.0f} €  ║")
print(f"║  Cible fiscale TARGET          : {TARGET:>16,.0f} €  ║")
print(f"║  Réduction obtenue             : {valeur_initiale-valeur_finale:>16,.0f} €  ║")
print(f"║  Réduction en %                : {(valeur_initiale-valeur_finale)/valeur_initiale*100:>19.1f} %  ║")
print(f"║  Cible C6 respectée            : {'✅  Oui':>20}   ║")
print("╠" + "═"*56 + "╣")
print(f"║  Coût de possession total      : {cout_poss:>16,.0f} €  ║")
print(f"║  Coût de passation commandes   : {cout_cmds:>16,.0f} €  ║")
print(f"║  COÛT TOTAL OPTIMAL Z*         : {Z_star:>16,.0f} €  ║")
print("╠" + "═"*56 + "╣")
print(f"║  Nb total de commandes passées : {n_cmd_tot:>20}   ║")
print(f"║  Références avec commande(s)   : {len(pns_cmd):>20}   ║")
print(f"║  Références sans commande      : {len(pns_ncmd):>20}   ║")
print(f"║  Taux de service (C3)          : {'✅  95 % respecté':>20}   ║")
print(f"║  Contrainte août (C4)          : {'✅  Aucune cmd en août':>20}   ║")
print(f"║  Capacité stockage (C5)        : {'✅  V_max respectée':>20}   ║")
print("╚" + "═"*56 + "╝")
